In [2]:
import os
from dotenv import load_dotenv
from pyannote.audio import Model
import torch
import torch.nn as nn
import onnx

load_dotenv(dotenv_path="../../.env")

repository = "pyannote/speaker-diarization-community-1"
token = os.getenv("HF_TOKEN")

example_waveform = torch.randn(
    1, 1, 160000,
    dtype=torch.float32,
)

## Segmentation Export

In [3]:
segmentation = Model.from_pretrained(
    repository,
    subfolder="segmentation",
    token=token,
    map_location="cpu",
    strict=False,
).eval()

In [ ]:
# -------------- EVALUATION -------------------
for name, module in segmentation.named_modules():
    if isinstance(module, torch.nn.InstanceNorm1d) or \
       isinstance(module, torch.nn.InstanceNorm2d) or \
       isinstance(module, torch.nn.InstanceNorm3d):
        print(
            name,
            type(module).__name__,
            "training =", module.training,
            "track_running_stats =", module.track_running_stats,
            "affine =", module.affine,
        )

for name, module in segmentation.named_modules():
    if isinstance(module, torch.nn.LSTM):
        print(
            name,
            "input_size =", module.input_size,
            "hidden_size =", module.hidden_size,
            "num_layers =", module.num_layers,
            "batch_first =", module.batch_first,
            "bidirectional =", module.bidirectional,
        )
        


duration = segmentation.specifications.duration
sample_rate = segmentation.audio.sample_rate

with torch.no_grad():
    y = model_to_check(example_waveform)

print("input:", example_waveform.shape)
print("output:", y.shape)
print("dtype:", y.dtype)
print("duration:", duration, "sampe_rate:", sample_rate)

input: torch.Size([1, 1, 160000])
output: torch.Size([1, 256])
dtype: torch.float32
duration: 5.0 sampe_rate: 16000


In [5]:
duration = segmentation.specifications.duration
sample_rate = segmentation.audio.sample_rate
num_samples = round(duration * sample_rate)
example_waveform = torch.randn(
    1, 1, num_samples,
    dtype=torch.float32,
)

with torch.no_grad():
    pytorch_output = segmentation(example_waveform)

print("sample_rate:", sample_rate)
print("duration:", duration)
print("num_samples:", num_samples)
print("input:", example_waveform.shape)
print("output:", pytorch_output.shape)

seg_destination = "../../models/segmentation.onnx"

with torch.no_grad():
    torch.onnx.export(
        segmentation,
        example_waveform,
        seg_destination,
        input_names=["waveform"],
        output_names=["powerset_scores"],
        opset_version=17,
        dynamo=False,
    )

print("Segment export complete")



sample_rate: 16000
duration: 10.0
num_samples: 160000
input: torch.Size([1, 1, 160000])
output: torch.Size([1, 589, 7])
Segment export complete


/tmp/ipykernel_55637/784683784.py:21: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [6]:
import onnx
import numpy as np
import onnxruntime as ort
import torch

model = onnx.load(seg_destination)
onnx.checker.check_model(model)

print("ONNX model is valid.")

for x in model.graph.input:
    print("INPUT:", x.name)

for x in model.graph.output:
    print("OUTPUT:", x.name)


# PyTorch
with torch.inference_mode():
    torch_output = segmentation(example_waveform)

# ONNX
session = ort.InferenceSession(
    seg_destination,
    providers=["CPUExecutionProvider"],
)

onnx_output = session.run(
    ["powerset_scores"],
    {
        "waveform": example_waveform.numpy(),
    },
)[0]

print("PyTorch:", torch_output.shape)
print("ONNX:   ", onnx_output.shape)

max_diff = np.max(
    np.abs(
        torch_output.numpy() - onnx_output
    )
)

mean_diff = np.mean(
    np.abs(
        torch_output.numpy() - onnx_output
    )
)

print("max abs difference:", max_diff)
print("mean abs difference:", mean_diff)

ONNX model is valid.
INPUT: waveform
OUTPUT: powerset_scores
PyTorch: torch.Size([1, 589, 7])
ONNX:    (1, 589, 7)
max abs difference: 2.0980835e-05
mean abs difference: 6.5941504e-06


## Embedding Export

In [7]:
embedding = Model.from_pretrained(
    repository,
    subfolder="embedding",
    token=token,
    map_location="cpu",
    strict=False,
).eval()

In [6]:
import inspect

with torch.inference_mode():
    example_fbank = embedding.compute_fbank(example_waveform)
# ---------------- EVALUATION ---------------
# print(embedding)

print("specifications:")
print(embedding.specifications)

print("audio sample rate:")
print(embedding.audio.sample_rate)

print("duration:")
print(embedding.specifications.duration)

# print(embedding.resnet)

with torch.inference_mode():
    x = embedding.compute_fbank(example_waveform)

print(x.shape)

print(inspect.getsource(type(embedding.resnet)))
source = inspect.getsource(embedding.resnet.forward)
print(source)

with torch.inference_mode():
    frames = embedding.resnet.forward_frames(example_fbank)

print("fbank:", example_fbank.shape)
print("resnet frames:", frames.shape)

specifications:
Specifications(problem=<Problem.REPRESENTATION: 3>, resolution=<Resolution.CHUNK: 2>, duration=5.0, min_duration=None, warm_up=(0.0, 0.0), classes=None, powerset_max_classes=None, permutation_invariant=False)
audio sample rate:
16000
duration:
5.0
torch.Size([1, 998, 80])
class ResNet(nn.Module):
    def __init__(
        self,
        block,
        num_blocks,
        m_channels=32,
        feat_dim=40,
        embed_dim=128,
        pooling_func="TSTP",
        two_emb_layer=True,
    ):
        super(ResNet, self).__init__()
        self.in_planes = m_channels
        self.feat_dim = feat_dim
        self.embed_dim = embed_dim
        self.stats_dim = int(feat_dim / 8) * m_channels * 8
        self.two_emb_layer = two_emb_layer

        self.conv1 = nn.Conv2d(
            1, m_channels, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(m_channels)
        self.layer1 = self._make_layer(block, m_channels, num_blocks[0], stride

In [8]:
import torch
import torch.nn as nn


class WeSpeakerONNX(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.resnet = model.resnet

    def forward(self, fbank, speaker_mask):
        _, speaker_embedding = self.resnet(
            fbank,
            weights=speaker_mask,
        )
        return speaker_embedding


# ---------------------------------------------------------
# 1. Create example input using the EMBEDDING model
# ---------------------------------------------------------

embedding.eval()

duration = embedding.specifications.duration
sample_rate = embedding.audio.sample_rate
num_samples = round(duration * sample_rate)

example_waveform = torch.randn(
    1,
    1,
    num_samples,
    dtype=torch.float32,
)


# ---------------------------------------------------------
# 2. Compute fbank
# ---------------------------------------------------------

with torch.inference_mode():
    example_fbank = embedding.compute_fbank(example_waveform)

print("fbank:", example_fbank.shape)
# [1, 998, 80]


# ---------------------------------------------------------
# 3. Determine ResNet temporal dimension
# ---------------------------------------------------------

with torch.inference_mode():
    example_frames = embedding.resnet.forward_frames(example_fbank)

print("resnet frames:", example_frames.shape)
# [1, 256, 10, 125]


# The pooling layer operates over the final temporal dimension.
num_embedding_frames = example_frames.shape[-1]

print("embedding frames:", num_embedding_frames)
# 125


# ---------------------------------------------------------
# 4. Create speaker mask
# ---------------------------------------------------------

example_mask = torch.ones(
    example_fbank.shape[0],
    num_embedding_frames,
    dtype=torch.float32,
)

print("speaker mask:", example_mask.shape)
# [1, 125]


# ---------------------------------------------------------
# 5. Create ONNX wrapper
# ---------------------------------------------------------

embedding_onnx = WeSpeakerONNX(embedding).eval()


# ---------------------------------------------------------
# 6. Export
# ---------------------------------------------------------
emb_dest = "../../models/embedding.onnx"

with torch.inference_mode():
    torch.onnx.export(
        embedding_onnx,
        (example_fbank, example_mask),
        emb_dest,
        input_names=[
            "fbank",
            "speaker_mask",
        ],

        output_names=[
            "embedding",
        ],

        opset_version=17,

        # Keep this false initially.
        # It uses the older exporter but is easier to debug.
        dynamo=False,
    )


# ---------------------------------------------------------
# 7. Validate ONNX file
# ---------------------------------------------------------

onnx_model = onnx.load(emb_dest)
onnx.checker.check_model(onnx_model)

print("ONNX model is valid.")

fbank: torch.Size([1, 498, 80])
resnet frames: torch.Size([1, 256, 10, 63])
embedding frames: 63
speaker mask: torch.Size([1, 63])
ONNX model is valid.


/tmp/ipykernel_55637/2766126374.py:92: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/home/arthur/projs/gossiper/.venv/lib/python3.14/site-packages/pyannote/audio/models/blocks/pooling.py:116: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if num_frames != num_weights:
/home/arthur/projs/gossiper/.venv/lib/python3.14/site-packages/pyannote/audio/models/embedding/wespeaker/resnet.py:430: TracerWarning: torch.tensor results